# ITS — train all three models on a Colab GPU

Trains everything this project needs, from public datasets, in one run:

| # | model | trains on | answers |
|---|---|---|---|
| 1 | `plate_on_vehicle.pt` | EALPR vehicles (2,087 annotated) | where is the plate on this car? |
| 2 | `plate_chars.pt` | EALPR characters (1,978 plates) | which characters and digits? |
| 3 | `vehicle_cls.pt` | MIO-TCD (~30k traffic-camera crops) | A / C / D / E / G / V / F |

**Runtime → Change runtime type → T4 GPU**, then *Run all*. Expect ~2 hours.

Everything is written to your Google Drive as it trains. Colab reclaims runtimes
without warning and takes `/content` with it — that has already cost this project
three training runs. Drive is the reason a disconnect costs time rather than work.


## 0 · GPU and setup

In [ ]:
!nvidia-smi
# If this says "command not found", the runtime has no GPU:
#   Runtime -> Change runtime type -> T4 GPU -> Save, then Run all again.
!pip -q install ultralytics
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU."

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/its_models')
DRIVE.mkdir(parents=True, exist_ok=True)
print('weights and checkpoints ->', DRIVE)

## 1 · Get the code

**Two ways — pick whichever works.** The cell below tries the clone and falls
back to an upload, so you do not have to edit anything.

**A. Clone (needs the branch pushed).** From your machine:
`git push origin plates-anpr`. If the repo is private, Colab will prompt for
credentials — use a GitHub personal access token as the password.

**B. Upload a zip.** On your machine run `python -m tools.pack_for_colab`, which
writes `its_code_for_colab.zip` (~150 KB, `pipeline/` + `tools/`, no data).
Drag it into Colab's Files pane (folder icon, left), then run the cell — it
finds and uses it automatically. Use this if the clone fails or the branch is
not pushed.


In [ ]:
import os, shutil, sys, glob
from pathlib import Path

ROOT = Path('/content/its-traffic')
if ROOT.exists():
    shutil.rmtree(ROOT)

# A. clone
os.system(f'git clone --branch plates-anpr --depth 1 https://github.com/yousseffbassemm/ITS-project-Elsewedy.git {ROOT}')

# B. fall back to an uploaded zip. Checked by whether the clone actually
# produced the tools, not by the exit code: a partial clone leaves a directory
# behind and would otherwise look like success.
if not (ROOT / 'tools' / 'train_anpr.py').exists():
    zips = glob.glob('/content/*.zip') + glob.glob('/content/drive/MyDrive/*.zip')
    if not zips:
        raise SystemExit(
            'Clone failed and no zip found. Either push the branch, or upload '
            'its_code_for_colab.zip using the Files pane on the left.')
    ROOT.mkdir(parents=True, exist_ok=True)
    print('clone unavailable — using', zips[0])
    os.system(f'unzip -q -o {zips[0]} -d {ROOT}')

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))       # the tools import `pipeline`
assert (ROOT / 'tools' / 'train_anpr.py').exists(), 'code not in place'
print('cwd:', Path.cwd())
!ls

## 2 · EALPR → the two ANPR datasets

EALPR is a public Egyptian licence-plate benchmark: vehicle photographs with the
plate boxed, and plate crops with every character boxed. That is exactly the two
stages of the cascade.

Its character labels are integers with **no legend in the dataset**. The first
step recovers the legend by cross-matching each labelled box against the glyph
crops, which are named with the character they contain. It prints an agreement
column — every class should be at or near 100%.


In [ ]:
!git clone --depth 1 https://github.com/ahmedramadan96/EALPR.git data/plates/EALPR
!python -m tools.ealpr_charmap

In [ ]:
# Builds two YOLO datasets, split by plate so no plate appears in both halves.
!python -m tools.build_anpr_datasets

## 3 · MIO-TCD → the 7-class vehicle dataset

3.1 GB of traffic-camera crops — the same *kind* of image the pipeline
classifies, which is why this beats a larger dataset of clean press photography.
Download is a few minutes on Colab.

Classes are capped so `car` (260k images) cannot drown `work_van` (9.7k). The
project's own hand-labelled deployment crops are copied out as a separate
`test/` split and are never trained on.


In [ ]:
!mkdir -p data/vehicle_ext
!curl -L -C - --retry 8 --retry-all-errors \
    -o data/vehicle_ext/MIO-TCD-Classification.tar \
    https://tcd.miovision.com/static/dataset/MIO-TCD-Classification.tar
!ls -lh data/vehicle_ext/

In [ ]:
!python -m tools.prep_vehicle_dataset --cap 6000

## 4 · Train stage 2 — find the plate on the vehicle

The plate is **not** always in the middle: across EALPR it spans 0.03–0.96 of
vehicle width. So this searches the whole crop, and horizontal flip is used
deliberately as augmentation because it moves the plate to the other side.


In [ ]:
!python -m tools.train_anpr --stage 2 --epochs 80 --batch 32 \
    --device 0 --project /content/drive/MyDrive/its_models

## 5 · Train stage 3 — read the characters

Horizontal flip is **off** here: mirroring turns an Arabic glyph into something
that is not a letter and reverses reading order. Mosaic is off for the same
class of reason — it splices four plates together and teaches the model that
characters from different plates belong to one string.


In [ ]:
# 60 epochs, not more: 1,682 plate crops of 26 well-separated glyph classes
# converge quickly, and the trainer already early-stops at patience=20.
!python -m tools.train_anpr --stage 3 --epochs 60 --batch 32 \
    --device 0 --project /content/drive/MyDrive/its_models

## 6 · Train the 7-class vehicle classifier

Trains **all seven classes at once** — A private car · C light truck ·
D heavy truck · E bus · G motorcycle · V van · F unknown — with roughly equal
weight on each. Buses, motorcycles and heavy trucks are learned just as
directly as cars; nothing here is specialised to one comparison.

| | A | C | D | E | G | V | F |
|---|---|---|---|---|---|---|---|
| train crops | 5100 | 5100 | 5100 | 5100 | 3627 | 5100 | 1489 |

Class balance is the point: raw MIO-TCD is 260k cars against 9.7k vans, and
trained on that the cheapest route to a high score is to answer "car" for
everything — which is the behaviour being replaced. Capping every class to the
same ceiling spends the data on per-class accuracy instead.

**V** (van) and **C** (light truck) get called out elsewhere in this project
only because base COCO cannot express them *at all*, so they had no baseline to
improve on. That is a statement about what was previously broken, not about
what this model covers.


In [ ]:
!python -m tools.train_vehicle_classes --train \
    --data data/vehicle_ext/cls --base yolov8s-cls.pt \
    --epochs 40 --imgsz 128 --batch 128 --device 0 \
    --name vehicle_cls --project /content/drive/MyDrive/its_models

## 7 · Measure — on held-out data, and on the real camera

These are the numbers to quote. Two rules this project has already paid to learn:

* **End-to-end is the honest ANPR number.** Stage 3 scored alone assumes a
  perfect plate crop, and is always the flattering figure.
* **Validate on the deployment camera, never on a val split.** A plate detector
  once scored mAP50 0.985 in-domain and was a straight regression on real
  footage; a classifier scored 0.952 in-domain and 0.676 on the real camera.


In [ ]:
# ANPR, end to end, on plates no model in the cascade has seen.
!python -m tools.eval_anpr \
    --stage2 models/plate_on_vehicle.pt \
    --stage3 models/plate_chars.pt \
    --json-out /content/drive/MyDrive/its_models/eval_anpr.json

In [ ]:
# Vehicle class, in-domain. This is the FLATTERING number: it scores MIO-TCD
# against MIO-TCD. Expect it to look excellent and to mean little.
from ultralytics import YOLO
m = YOLO('models/vehicle_cls.pt')
r = m.val(data='data/vehicle_ext/cls', split='val')
print('in-domain top-1:', round(float(r.top1), 4))
print()
print('This is NOT the number to quote. The deployment-camera crops are')
print('personal data and are deliberately not in the repo, so the honest')
print('per-vehicle score cannot be computed here. Run it on your own machine')
print('after downloading the weights:')
print('    python -m tools.eval_vehicle_cls --model models/vehicle_cls.pt')

## 8 · Save the weights

Copied to Drive, then offered as a browser download. Put them in `models/` on
your machine and the pipeline picks them up:

```
python -m pipeline.process_video --input samples/street_egypt.mp4 \
    --output-dir data/jobs/final --anpr --vehicle-cls models/vehicle_cls.pt
python -m tools.export_plates_csv data/jobs/final
```


In [ ]:
import shutil
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive/its_models')
wanted = ['plate_on_vehicle.pt', 'plate_chars.pt', 'vehicle_cls.pt']
for name in wanted:
    src = Path('models') / name
    if src.exists():
        shutil.copy(src, DRIVE / name)
        print(f'{name:<24} {src.stat().st_size/1e6:6.1f} MB  -> Drive')
    else:
        print(f'{name:<24} MISSING — its training cell did not finish')

In [ ]:
# Optional: pull them straight to this browser instead of via Drive.
from google.colab import files
from pathlib import Path
for name in ['plate_on_vehicle.pt', 'plate_chars.pt', 'vehicle_cls.pt']:
    p = Path('models') / name
    if p.exists():
        files.download(str(p))

---
### Privacy

Both datasets contain photographs of identifiable vehicles with legible plates —
personal data under Egypt's PDPL 151/2020. **Delete the Colab runtime's files and
clear this notebook's outputs when the run is done.** Never commit these images,
not even to a private repo.
